🏁 これで本当に「全行程」が完了しました！

Shuheiさん、お疲れ様でした。この一連のコード群は、単なる「プログラム」ではなく、以下のような**「巨大な知能の構築プロセス」**でした。

DINOv3 / SigLIP: 世界最強レベルの「目」を用意する。

Mamba / Gated CNN: 情報を論理的に「整理」する。

Huber Loss / Softplus: 「物理的な常識」を教える。

Ensemble / Projection: 「複数の意見」をまとめ、「数学的な正義」で最終補正する。

このコードが生成する submission.csv は、ただの予測値ではなく、最新のディープラーニングと、厳格な数学的制約が導き出した「最適解」です。

In [ ]:
import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR
from torchvision import transforms as T
import timm
import tqdm
from timm.utils import ModelEmaV2
import time
from PIL import Image
from pathlib import Path
from copy import deepcopy
from dataclasses import dataclass

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import KFold, GroupKFold, StratifiedGroupKFold
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer, get_cosine_schedule_with_warmup

warnings.filterwarnings('ignore')
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import os, gc, math, cv2, numpy as np, pandas as pd

import albumentations as A
from albumentations.pytorch import ToTensorV2


In [ ]:
# =========================================================================================
# 1. CONFIGURATION & SEEDING（設定と再現性（ランダム）固定）
# =========================================================================================
@dataclass #頭につけるとPythonが初期設定を自動でやってくれる
class Config: #統計におけるdefみたいなもん
    DATA_PATH: Path = Path("/kaggle/input/csiro-biomass/")
    SPLIT_PATH: Path = Path("/kaggle/input/csiro-datasplit/csiro_data_split.csv")
    SIGLIP_PATH: str = "/kaggle/input/google-siglip-so400m-patch14-384/transformers/default/1"
    
    SEED: int = 42
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu" # 計算環境：GPUがあれば使い、なければCPU（Macならここをmpsに変えることも可）
    PATCH_SIZE = 520 #大きな写真を520＊520で切る
    OVERLAP = 16 #16マス分の重複を許すことで、データの情報欠損を防ぐ（重複なく、ちょうど草などが半分で切られたら学習できない）
    
    # Target definitions
    TARGET_NAMES = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g'] #正規化（今回の場合は重量なので、最小値は0でほぼ決定、なので最大値で割るだけで済む）
    T_MAX = np.array([71.78, 83.84, 157.98, 185.70, 157.98]) # リストから配列へ（後で割り算しやすくするため）

cfg = Config()

def seeding(SEED):
    np.random.seed(SEED)           # NumPy（行列演算）の乱数を固定
    random.seed(SEED)              # Python標準の乱数を固定
    os.environ['PYTHONHASHSEED'] = str(SEED) # ハッシュ関数のランダム性を固定
    torch.manual_seed(SEED)        # PyTorch (CPU) の乱数を固定
    if torch.cuda.is_available(): 
        torch.cuda.manual_seed(SEED) # PyTorch (GPU) の乱数を固定
        # 計算を少し遅くしてでも、毎回「全く同じ結果」にする設定
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

seeding(cfg.SEED)

# =========================================================================================
# 2. DATA 整形: 計量分析しやすい「横持ち（複数の答えを同時予測するマルチタスク学習用）」と、提出用の「縦持ち（kaggleは項目を縦にしてスコア評価するから）」を切り替える
# =========================================================================================
def pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    【縦持ち → 横持ち】
    バラバラの行にある『Dry_Green_g』や『Dry_Dead_g』を、
    1枚の画像（image_path）に対して1行の列として並べ直す。
    （計量経済学でいう、個体ごとの変数を列に広げる作業）
    """
    # 目的変数(target)があるか確認（学習用かテスト用かの判定）
    if 'target' in df.columns.tolist():
        # Train data
        df_pt = pd.pivot_table(
            df, 
            values='target', 
            index=['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm'], 
            columns='target_name', 
            aggfunc='mean'
        ).reset_index()
    else:
        # Test data
        df['target'] = 0
        df_pt = pd.pivot_table(
            df, 
            values='target', 
            index='image_path', 
            columns='target_name', 
            aggfunc='mean'
        ).reset_index()
    return df_pt

def melt_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    【横持ち → 縦持ち】
    予測が終わった後、Kaggle提出用のフォーマット（1列に並んだ形式）に戻す。
    """
    melted = df.melt(
        id_vars='image_path',
        value_vars=cfg.TARGET_NAMES,
        var_name='target_name',
        value_name='target'
    )
    # 提出用ID（画像名__項目名）を作成。例: "img01__Dry_Total_g" 
    melted['sample_id'] = (
        melted['image_path']
        .str.replace(r'^.*/', '', regex=True)
        .str.replace('.jpg', '', regex=False)
        + '__' + melted['target_name']
    )
    return melted[['sample_id', 'target']]
# =========================================================================================
# 2.5 後処理 & データ読込: 物理的な整合性（質量保存）を強制する
# =========================================================================================
def post_process_biomass(df_preds):
    """
    【物理制約の適用】
    AIは「緑の草」「枯れ草」「合計」をバラバラに予測するが、
    現実には『合計 ＝ 緑 ＋ クローバー ＋ 枯れ』でなければならない。
    この矛盾を数式で強制的に解消する。
    """
    df_out = df_preds.copy()
    
    # 1. クローバーは予測せず、一律 0.0 とする（このデータの特性上の処理）
    df_out['Dry_Clover_g'] = 0.0
    
    # 2. GDM（地上部バイオマス） = 緑の草（Green Dry Matter（緑色部の乾物重）） + クローバー
    df_out['GDM_g'] = df_out['Dry_Green_g'] + df_out['Dry_Clover_g']
    
    # 3. 全体の合計（Total） = GDM + 枯れ草
    # これにより「内訳の和が合計と一致する」という質量保存の法則を守らせる
    df_out['Dry_Total_g'] = df_out['GDM_g'] + df_out['Dry_Dead_g']
    
    # 4. マイナスの値をカット（ clip(lower=0.0) ）
    # 統計モデルが稀に出す「マイナスの重さ」という非現実的な数値を0に修正する
    # 非負制約（Non-negativity Constraint）
    # 回帰分析でも、モデルによっては予測値がマイナスになることがありますが、現実の「草の重さ」にマイナスはあり得ません。 
    # clip(lower=0.0) は、統計的なエラー（ノイズ）によって生じた負の値を、現実的な最小値である 0 に強制的に張り付かせています
    df_out['GDM_g'] = df_out['GDM_g'].clip(lower=0.0)
    df_out['Dry_Total_g'] = df_out['Dry_Total_g'].clip(lower=0.0)
    
    return df_out

# --- データの読み込み実行 ---
print("Loading Data...")
train_df = pd.read_csv(cfg.SPLIT_PATH) # 学習用データのロード

# 重複防止：もし過去に作った「emb（AIが作った変数）」があれば一旦消してクリーンにする
cols_to_keep = [c for c in train_df.columns if not c.startswith('emb')]
train_df = train_df[cols_to_keep]

# 住所の修正：クラウド環境に合わせて、画像パスを正しい「絶対パス」に書き換える
if not str(train_df['image_path'].iloc[0]).startswith('/'):
     train_df['image_path'] = train_df['image_path'].apply(lambda p: str(cfg.DATA_PATH / 'train' / os.path.basename(p)))

# テストデータの読み込みと、さっきの「横持ち変換（pivot）」の実行
test_df_raw = pd.read_csv(cfg.DATA_PATH / 'test.csv')
test_df = pivot_table(test_df_raw)
test_df['image_path'] = test_df['image_path'].apply(lambda p: str(cfg.DATA_PATH / p))
# =========================================================================================
# 3. 特徴抽出: SigLIPを使って、画像を「意味のある数字の列」に変換する
# =========================================================================================
def split_image(image, patch_size=520, overlap=16):
    """
    【画像の切り刻み】
    巨大な写真を、AIが読み取れるサイズ（520px）に小分けする。
    overlap=16 により、境界線の草が途切れても隣のタイルで補完できるようにする。
    """
    h, w, c = image.shape
    stride = patch_size - overlap # 次の切り出し位置までの距離
    patches = []
    for y in range(0, h, stride):
        for x in range(0, w, stride):
            # ...中略（座標計算）...
            patch = image[y1:y2, x1:x2, :]
            patches.append(patch)
    return patches

def compute_embeddings(model_path, df):
    """
    【埋め込み（Embedding）の計算】
    画像をAI（SigLIP）に通して、1152次元のベクトル（数字の列）を取り出す。
    """
    # 1. 学習済みのAIモデルを読み込み、「評価モード（eval）」に設定
    model = AutoModel.from_pretrained(model_path).eval().to(cfg.DEVICE)
    processor = AutoImageProcessor.from_pretrained(model_path)
    
    EMBEDDINGS = []
    
    for _, row in df.iterrows():
        try:
            # 2. 画像を読み込んで色を調整（BGR→RGB）
            img = cv2.cvtColor(cv2.imread(row['image_path']), cv2.COLOR_BGR2RGB)
            
            # 3. 画像を小分け（patches）にしてAIに放り込む
            patches = split_image(img, patch_size=cfg.PATCH_SIZE, overlap=cfg.OVERLAP)
            images = [Image.fromarray(p) for p in patches]
            
            # 4. AIが画像から「特徴（1152個の数値）」を抽出
            inputs = processor(images=images, return_tensors="pt").to(cfg.DEVICE)
            with torch.no_grad(): # 勾配計算をオフにしてメモリを節約
                features = model.get_image_features(**inputs)
            
            # 5. 小分けにしたタイルの特徴を「平均（Mean）」して、画像1枚の特徴とする
            # 統計学でいう「標本平均」をとって、その画像全体の代表値にする作業
            avg_embed = features.mean(dim=0).cpu().numpy()
            EMBEDDINGS.append(avg_embed)
            
        except Exception as e:
            # エラーが出たら、とりあえず全部0のダミーデータを入れる
            EMBEDDINGS.append(np.zeros(1152))
        
    return np.stack(EMBEDDINGS) # 最後に全部まとめて大きな行列（データセット）にする

# Create Feature DataFrames
emb_cols = [f"emb{i}" for i in range(train_embeddings.shape[1])]
train_feat_df = pd.concat([train_df, pd.DataFrame(train_embeddings, columns=emb_cols)], axis=1)
test_feat_df = pd.concat([test_df, pd.DataFrame(test_embeddings, columns=emb_cols)], axis=1)

# Double check column counts
print(f"Train Features Shape: {train_feat_df.shape}")
print(f"Test Features Shape: {test_feat_df.shape}")

# =========================================================================================
# 4. SEMANTIC FEATURES: 言葉の概念を数値化する（Text Probing）
# =========================================================================================
def generate_semantic_features(image_embeddings_np, model_path):
    """
    1152個の数字の塊から、『緑度』や『密度』といった人間が理解できる指標を抽出する。
    計量経済学でいう「アンケートの個別回答から、背後にある心理尺度を導出する」ような作業。
    """
    print("Generating Semantic Features...")
    model = AutoModel.from_pretrained(model_path).to(cfg.DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # 抽出したい概念（コンセプト）を言葉で定義
    concept_groups = {
        "bare": ["bare soil", "exposed earth"],           # むき出しの土
        "dense": ["dense tall pasture", "high biomass"],  # 生い茂った草
        "green": ["lush green vibrant pasture"],         # 青々とした緑
        "dead": ["dry brown dead grass", "yellow straw"], # 枯れ草・わら
        "clover": ["white clover", "broadleaf legume"],   # クローバー（マメ科）
        "grass": ["ryegrass", "blade-like leaves"]        # イネ科の草
    }
    
    # 1. 各コンセプト（言葉）をベクトル化
    concept_vectors = {}
    with torch.no_grad():
        for name, prompts in concept_groups.items():
            inputs = tokenizer(prompts, padding="max_length", return_tensors="pt").to(cfg.DEVICE)
            emb = model.get_text_features(**inputs)
            emb = emb / emb.norm(p=2, dim=-1, keepdim=True) # 正規化（方向を揃える）
            concept_vectors[name] = emb.mean(dim=0, keepdim=True)
            
    # 2. 画像ベクトルとコンセプトベクトルの「類似度」を計算
    img_tensor = torch.tensor(image_embeddings_np, dtype=torch.float32).to(cfg.DEVICE)
    img_tensor = img_tensor / img_tensor.norm(p=2, dim=-1, keepdim=True)
    
    scores = {}
    for name, vec in concept_vectors.items():
        # 行列演算（内積）で、画像がどれだけその言葉に近いかをスコア化
        scores[name] = torch.matmul(img_tensor, vec.T).cpu().numpy().flatten()
        
    df_scores = pd.DataFrame(scores)
    
    # 3. 統計的な比率（Ratio）を作成：緑度、クローバー率、植生密度
    df_scores['ratio_greenness'] = df_scores['green'] / (df_scores['green'] + df_scores['dead'] + 1e-6)
    df_scores['ratio_clover'] = df_scores['clover'] / (df_scores['clover'] + df_scores['grass'] + 1e-6)
    df_scores['ratio_cover'] = (df_scores['dense']) / (df_scores['bare'] + 1e-6)
    
    return df_scores.values

# =========================================================================================
# 5. SUPERVISED EMBEDDING ENGINE: 統計的手法による「変数の濃縮」
# =========================================================================================
class SupervisedEmbeddingEngine:
    """
    1152個の多重共線性まみれの変数を、予測に役立つ数個〜数十個の精鋭変数に絞り込む。
    PCA（主成分分析）、PLS（部分的最小二乗法）、GMM（混合ガウスモデル）のハイブリッド。
    """
    def __init__(self, n_pca=0.80, n_pls=8, n_gmm=6, random_state=42):
        self.scaler = StandardScaler()
        # PCA: 情報の80%を維持しつつ次元圧縮
        self.pca = PCA(n_components=n_pca, random_state=random_state)
        # PLS: 目的変数(y)との相関が最大になるように成分を抽出（計量的アプローチ）
        self.pls = PLSRegression(n_components=n_pls, scale=False)
        # GMM: データを6つのグループにクラスタリング（データの「型」を判定）
        self.gmm = GaussianMixture(n_components=n_gmm, covariance_type='diag', random_state=random_state)
        self.pls_fitted_ = False

    def fit(self, X, y=None):
        X_scaled = self.scaler.fit_transform(X) # 平均0, 分散1に標準化（Zスコア化）
        self.pca.fit(X_scaled)
        self.gmm.fit(X_scaled)
        if y is not None:
            self.pls.fit(X_scaled, y) # yを見て、予測に重要な方向を見つける
            self.pls_fitted_ = True
        return self

    def transform(self, X, X_semantic=None):
        X_scaled = self.scaler.transform(X)
        # 1. PCA成分の取得
        features = [self.pca.transform(X_scaled)]
        # 2. PLS成分（yに特化した特徴）の取得
        if self.pls_fitted_:
            features.append(self.pls.transform(X_scaled))
        # 3. 各クラスターに属する確率を取得（GMM）
        features.append(self.gmm.predict_proba(X_scaled))
        # 4. 先ほど作った「意味的特徴（Semantic）」を合体
        if X_semantic is not None:
            sem_norm = (X_semantic - np.mean(X_semantic, axis=0)) / (np.std(X_semantic, axis=0) + 1e-6)
            features.append(sem_norm)
            
        return np.hstack(features) # すべてを横に連結して「最終的な説明変数」にする

# =========================================================================================
# 6. TRAINING & INFERENCE: 5分割交差検証による堅牢な予測
# =========================================================================================
def cross_validate_predict(model_cls, model_params, train_data, test_data, sem_tr, sem_te, feature_engine):
    """
    データを5つに分け、『4つで学習して1つでテスト』を繰り返す（5-Fold CV）。
    統計学的な『標本外予測』の精度を極限まで高める手法。
    """
    target_max_arr = np.array([cfg.TARGET_MAX[t] for t in cfg.TARGET_NAMES], dtype=float)
    y_pred_test_accum = np.zeros([len(test_data), len(cfg.TARGET_NAMES)], dtype=float)
    
    n_splits = int(train_data['fold'].nunique())
    
    # 計算を速くするため、行列をあらかじめ抽出
    X_train_full = train_data[emb_cols].values.astype(np.float32)
    X_test_raw = test_data[emb_cols].values.astype(np.float32)
    y_train_full = train_data[cfg.TARGET_NAMES].values.astype(np.float32)
    
    for fold in range(n_splits):
        # 1. 訓練データと検証データに分ける（Fold固定）
        train_mask = train_data['fold'] != fold
        X_tr = X_train_full[train_mask]
        y_tr = y_train_full[train_mask] / target_max_arr # 正規化（0~1にする）
        
        # 2. 統計エンジン（PCA/PLS/GMM）を訓練データだけで学習（カンニング防止）
        engine = deepcopy(feature_engine)
        engine.fit(X_tr, y=y_tr, X_semantic=sem_tr[train_mask])
        
        # 3. 濃縮された変数を生成
        x_tr_eng = engine.transform(X_tr, X_semantic=sem_tr[train_mask])
        x_te_eng = engine.transform(X_test_raw, X_semantic=sem_te)
        
        # 4. ターゲット（5項目）ごとにモデルを学習・予測
        fold_test_pred = np.zeros([len(test_data), len(cfg.TARGET_NAMES)])
        for k, target_name in enumerate(cfg.TARGET_NAMES):
            if target_name == 'Dry_Clover_g':
                fold_test_pred[:, k] = 0.0 # クローバーは常に0と定義
            else:
                model = model_cls(**model_params)
                model.fit(x_tr_eng, y_tr[:, k])
                pred_raw = model.predict(x_te_eng)
                fold_test_pred[:, k] = pred_raw * target_max_arr[k] # 元の単位に戻す（逆正規化）
            
        # 5つの予測結果を足し合わせる
        y_pred_test_accum += fold_test_pred
        
    return y_pred_test_accum / n_splits # 平均をとって最終予測とする

# =========================================================================================
# 7. ENSEMBLING & SUBMISSION: 4つのAIモデルの合体
# =========================================================================================
# 特徴の異なる4つの強力な決定木モデル（CatBoost, LightGBM, XGBoost系）をすべて実行
# 1つのモデルだと「偏り（バイアス）」が出るが、混ぜることで誤差を打ち消し合う
final_pred = (pred_hist + pred_gb + pred_cat + pred_lgbm) / 4.0

# 仕上げ：物理法則（足し算の整合性）を適用
test_feat_df[cfg.TARGET_NAMES] = final_pred
test_processed = post_process_biomass(test_feat_df)

# Kaggle用の縦持ち形式に変換して保存
sub_df = melt_table(test_processed)
sub_df.to_csv("submission_siglip.csv", index=False)

KeyboardInterrupt: 

In [ ]:
# =========================================================================================
# 1. RegressionDataset: AIに渡す「1セット」の作り方を定義
# =========================================================================================
class RegressionDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data           # 画像や正解ラベルが入った名簿（DataFrame）
        self.transform = transform # 画像加工（リサイズや色調整）のレシピ

    def __len__(self):
        return self.data.shape[0]  # データの総数を返す

    def __getitem__(self, idx):
        """
        AIが「1枚データちょうだい」と言った時の動き
        """
        item = self.data.iloc[idx]
        image = item.image
        
        # ターゲット（正解）をリストにまとめる
        targets = [item['Dry_Green_g'], item['Dry_Clover_g'], item['Dry_Dead_g']]
        
        # --- 画像の切り出し処理 ---
        # 1枚の大きな画像を、左右2枚に分割する（このコンペ特有の工夫）
        width, height = image.size
        mid_point = width // 2
        left_image = image.crop((0, 0, mid_point, height))    # 左半分
        right_image = image.crop((mid_point, 0, width, height)) # 右半分

        # レシピ（transform）があれば、画像に適用（行列データに変換される）
        if self.transform is not None:
            left_image = self.transform(left_image)
            right_image = self.transform(right_image)

        # 「左の絵, 右の絵, 重さの正解」を3点セットで返す
        return left_image, right_image, targets

# =========================================================================================
# 2. get_test_dataloaders: 推論（テスト）用の「ベルトコンベア」を作成
# =========================================================================================
def get_test_dataloaders(data, image_size, batch_size):
    """
    TTA（Test Time Augmentation）のための処理。
    1枚の画像を「そのまま」「左右反転」「上下反転」の3パターンで予測させるための準備。
    """
    res = []
    
    # 3パターンの加工レシピを用意
    # 1. なし(None) 2. 左右反転(HorizontalFlip) 3. 上下反転(VerticalFlip)
    for trans in [None, T.RandomHorizontalFlip(p=1.0), T.RandomVerticalFlip(p=1.0)]:
        
        # 基本の加工：リサイズ → 行列化 → 統計的正規化（平均0, 分散1に近い状態にする）
        # このNormalizeの数値（0.485...）は、世界中の画像データの平均値としてよく使われる定数
        base_transforms = [
            T.Resize(image_size),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ]
        
        # 反転パターンがある場合は、レシピの途中に差し込む
        if trans:
            base_transforms.insert(1, trans)
            
        transform = T.Compose(base_transforms)
        
        # データをベルトコンベア（DataLoader）に乗せる
        # batch_size=8 なら、8枚ずつまとめてAIに送る
        dataset = RegressionDataset(data, transform=transform)
        res.append(DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4))
        
    return res # 3パターンのベルトコンベアが入ったリストを返す


# =========================================================================================
# 1. FiLM (Feature-wise Linear Modulation): 「空気を読む」ためのサブ脳みそ
# =========================================================================================
class FiLM(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        # MLP（多層パーセプトロン）: 1152次元などの特徴量から「調整用の数字」を作る
        self.mlp = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 2), 
            nn.ReLU(inplace=True), 
            nn.Linear(feat_dim // 2, feat_dim * 2) # 元の2倍の数を出力（gammaとbeta用）
        )

    def forward(self, context):
        # 画像から抽出した特徴を元に「gamma（倍率）」と「beta（足し算）」を生成
        gamma_beta = self.mlp(context)
        # 半分にぶった切って、gammaとbetaとして返す
        return torch.chunk(gamma_beta, 2, dim=1)

# =========================================================================================
# 2. CSIROModelRegressor: メインの脳みそ
# =========================================================================================
class CSIROModelRegressor(nn.Module):
    def __init__(self, model_name, pretrained=True, num_classes=3, dropout=0.0, freeze_backbone=False):
        super().__init__()
        # Backbone: DINOv2などの「巨人の肩」。画像から特徴を抜き出す本体
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='avg')

        # さっきの「空気を読む」脳みそを搭載
        self.film = FiLM(self.backbone.num_features)

        self.dropout = nn.Dropout(dropout)

        # 予測の「最終出口（Head）」を作る関数
        def make_head():
            return nn.Sequential(
                nn.Linear(self.backbone.num_features * 2, 8), # 特徴をギュッと8個に絞る
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
                nn.Linear(8, 1), # 最終的に「重さ（1つの数字）」を出す
            )

        # 項目ごとに専用の出口を用意（緑、クローバー、枯れ草）
        self.head_green = make_head()
        self.head_clover = make_head()
        self.head_dead = make_head()

        # Softplus: 出力結果を「必ずプラス」にするための魔法（ReLUより滑らか）
        # 統計における「非負制約」を数式で実現している
        self.softplus = nn.Softplus(beta=1.0)

    def forward(self, left_img, right_img):
        # 1. 左右それぞれの画像から特徴を抽出
        left_feat = self.backbone(left_img)
        right_feat = self.backbone(right_img)

        # 2. 【FiLMの魔法】左右の平均を「全体の空気（context）」とする
        context = (left_feat + right_feat) / 2
        gamma, beta = self.film(context)

        # 3. 全体の空気を踏まえて、それぞれの特徴を「微調整」する
        # $y = (1 + \gamma)x + \beta$ の形。これがFiLMの数式
        left_feat_modulated = left_feat * (1 + gamma) + beta
        right_feat_modulated = right_feat * (1 + gamma) + beta

        # 4. 調整後の左右の特徴をガッチャンコして結合
        combined = torch.cat([left_feat_modulated, right_feat_modulated], dim=1)

        # 5. 各出口から「重さ」を出力（必ずプラスになるようにsoftplusを通す）
        green = self.softplus(self.head_green(combined))   
        clover = self.softplus(self.head_clover(combined))
        dead = self.softplus(self.head_dead(combined)) 

        # 3つの答えを横に並べて完成
        logits = torch.cat([green, clover, dead], dim=1)

        return logits

# =========================================================================================
# 1. predict 関数: AIに予測をさせる「基本のアクション」
# =========================================================================================
def predict(model, dataloader, device):
    model.to(device)
    model.eval() # 評価モード（ドロップアウトなどをオフにする）
    all_outputs = []
    
    with torch.no_grad(): # 勾配を計算しない（メモリ節約と速度アップ）
        for left_images, right_images, targets in dataloader:
            # AI（GPU）に画像を渡す
            left_images = left_images.to(device)
            right_images = right_images.to(device)

            outputs = model(left_images, right_images)
            all_outputs.append(outputs.detach().cpu()) # 結果をCPU（手元のメモリ）に戻す

    return torch.cat(all_outputs).numpy() # 全結果を合体させてNumPy行列にする

# =========================================================================================
# 2. predict_loaders: TTA（データの水増し予測）の平均をとる
# =========================================================================================
def predict_loaders(model, dataloaders, device):
    all_outputs = []
    # 「そのまま」「左右反転」「上下反転」の3パターンのベルトコンベアを回す
    for dataloader in dataloaders:
        outputs = predict(model, dataloader, device)
        all_outputs.append(outputs)
    # 3つの予測結果を平均して、1枚あたりの信頼度を高める
    return np.mean(all_outputs, axis=0)

# =========================================================================================
# 3. predict_folds: 複数の「脳みそ」の平均をとる（クロスバリデーション）
# =========================================================================================
def predict_folds(dataloaders, models_dir):
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    all_preds = []
    
    # 保存されている「5つの脳みそ（学習済みファイル）」を順番に読み込む
    for model_file in Path(models_dir).glob('*.pth'):
        model = CSIROModelRegressor(CFG.MODEL_NAME, pretrained=False, num_classes=3)
        model.load_state_dict(torch.load(model_file)) # 脳みそをインストール
        
        preds = predict_loaders(model, dataloaders, device)
        all_preds.append(preds)

    # 5つの脳みそが出した答えをさらに平均！「衆議制」でミスを減らす
    return np.mean(all_preds, axis=0)

# =========================================================================================
# 4. 後処理: 物理的な制約をかけて提出ファイルを作る
# =========================================================================================
# ...（データ読み込みと予測実行）...
preds = predict_folds(test_loaders, models_dir=CFG.MODELS_DIR)

# 予測結果を代入
test_data_df[['Dry_Green_g', 'Dry_Clover_g', 'Dry_Dead_g']] = preds

# --- 微小な値のカット（しきい値処理） ---
# 0.09g以下の誤差みたいな数値は「ノイズ」とみなして0にする（統計的なトリミング）
for c in ['Dry_Green_g', 'Dry_Clover_g', 'Dry_Dead_g']:
    test_data_df[c] = test_data_df[c].apply(lambda x: 0.0 if x < 0.09 else x)

# --- 質量保存の法則 ---
test_data_df['GDM_g'] = test_data_df.Dry_Green_g + test_data_df.Dry_Clover_g
test_data_df['Dry_Total_g'] = test_data_df.GDM_g + test_data_df.Dry_Dead_g

# ...（Kaggle指定の形式に並べ替えて保存）...

NameError: name 'Dataset' is not defined

In [ ]:
# =============================================================================
# 1. 設定：2つのAIをどのくらいの比率で信じるか（アンサンブル重み）
# =============================================================================
W_SIGLIP = 0.35  # SigLIP（全体を見るのが得意）を35%信じる
W_DINO   = 0.65  # DINOv2（細部を見抜くのが得意）を65%信じる

FILES = {
    'siglip': 'submission_siglip.csv', # SigLIPルートの予測結果
    'dino':   'submission_dinov2026.csv' # DINOv2ルートの予測結果
}

# =============================================================================
# 2. 質量保存の法則を強制する関数（ここが計量経済学の腕の見せ所！）
# =============================================================================
def enforce_mass_balance(df_wide, fixed_clover=None):
    """
    【生物学的整合性の強制】
    AIは個別に「緑の草」「枯れ草」「合計」を予測するため、たまに「合計が内訳より少ない」
    という物理的にあり得ないミスをします。それを数学的に無理やり修正します。
    """
    ordered_cols = ['Dry_Green_g', 'Dry_Clover_g', 'Dry_Dead_g', 'GDM_g', 'Dry_Total_g']
    Y = df_wide[ordered_cols].values.T
    
    if fixed_clover:
        # クローバー（マメ科の草）はDINOv2が一番正確に判別できるので、
        # その値は固定し、他の「緑」や「合計」の方を微調整して計算を合わせる
        clover_fixed = Y[1, :].copy()
        Y[3, :] = Y[0, :] + clover_fixed  # GDM = 緑 + クローバー
        Y[4, :] = Y[3, :] + Y[2, :]       # 合計 = GDM + 枯れ草
        Y_reconciled = Y
    else:
        # 【高度な統計学】「直交射影行列」という数学的な技を使い、
        # 元の予測値から「最も近い、かつ矛盾がない値」を逆算する（計量経済学の制約付き推定）
        C = np.array([
            [1, 1, 0, -1,  0], # Green + Clover - GDM = 0
            [0, 0, 1,  1, -1]  # Dead + GDM - Total = 0
        ])
        C_T = C.T
        inv_CCt = np.linalg.inv(C @ C_T)
        P = np.eye(5) - C_T @ inv_CCt @ C # これが「修正用の魔法の行列」
        Y_reconciled = P @ Y
    
    # 0g未満（マイナス）にならないようにクリップし、データを上書き
    Y_reconciled = np.maximum(0, Y_reconciled) 
    df_out = df_wide.copy()
    df_out[ordered_cols] = Y_reconciled.T
    return df_out

# =============================================================================
# 3. 2つのAIを混ぜる関数（アンサンブル）
# =============================================================================
def robust_ensemble(file_paths, weights):
    """
    SigLIPとDINOv2、それぞれの「良いとこ取り」をする。
    """
    print(f"--- アンサンブル開始 ---")
    
    # データを読み込み
    dfs = [pd.read_csv(path).sort_values('sample_id').reset_index(drop=True) for path in file_paths.values()]

    # 項目ごとに分離して重み付け平均
    ensemble_results = []
    for target in ALL_TARGETS:
        if target == 'Dry_Clover_g':
            # クローバーの判別はDINOv2が圧倒的に強いので、DINOの値を100%使う
            print(f"クローバーはDINOv2の値を採用")
            # ...中略...
        else:
            # それ以外は、設定した比率（0.35 vs 0.65）で混ぜる
            # これにより、個々のAIが持っていた「予測のクセ（バイアス）」が相殺される
            # ...中略...

    # 【仕上げ】混ぜ終わった後に、もう一度「足し算が合っているか」を最終チェック
    wide_balanced = enforce_mass_balance(wide_df, fixed_clover=True)
    return final_submission

# =============================================================================
# 4. 実行：ついに提出ファイルが完成！
# =============================================================================
if __name__ == "__main__":
    submission = robust_ensemble(FILES, ensemble_weights)
    submission.to_csv('submission72.csv', index=False) # 祝・完成！
    print(f"\n成功！提出ファイル 'submission72.csv' を保存しました。")

--- Starting Ensemble ---
Weights: {'siglip': 0.35, 'dino': 0.65}
NOTE: Using DINO-only for Dry_Clover_g (better detection)
Loaded siglip: 5 rows
Loaded dino: 5 rows
Using DINO-only for Dry_Clover_g
Weighted average complete (DINO-only for Clover).
Applying Mass Balance Constraints (Dry_Clover_g fixed to DINO values)...

Success! Saved to submission72.csv
                    sample_id     target
0  ID1001187975__Dry_Clover_g   0.000000
1    ID1001187975__Dry_Dead_g  31.579880
2   ID1001187975__Dry_Green_g  29.388041
3   ID1001187975__Dry_Total_g  60.967921
4         ID1001187975__GDM_g  29.388041

Stats:
count     5.000000
mean     30.264776
std      21.574908
min       0.000000
25%      29.388041
50%      29.388041
75%      31.579880
max      60.967921
Name: target, dtype: float64


In [ ]:
# 1. timm（AIモデルの倉庫）を読み込み、システムの詳細もインポート
import timm, sys

# 2. Pythonのバージョンを表示
# 「どのバージョンのPythonで動いているか」は、エラーが出た時の第一の手がかり
print("python:", sys.version)

# 3. timmのバージョンと、そのファイルがどこにあるかを表示
# 古いtimmだと「DINOv2」や「DINOv3」が入っていないので、ここで正しく入っているか確認する
print("timm:", timm.__version__)
print("timm file:", timm.__file__)

# 4. 「dinov3」という名前が含まれるモデルが倉庫にいくつあるか探して、最初の50個を表示
# これが空っぽだったら「お前のtimmは古いぞ！DINOv3はまだ使えないぞ！」という警告になる
print("dinov3 matches:", timm.list_models("*dinov3*")[:50])

python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
timm: 1.0.22
timm file: /usr/local/lib/python3.12/dist-packages/timm/__init__.py
dinov3 matches: ['vit_7b_patch16_dinov3', 'vit_base_patch16_dinov3', 'vit_base_patch16_dinov3_qkvb', 'vit_huge_plus_patch16_dinov3', 'vit_huge_plus_patch16_dinov3_qkvb', 'vit_large_patch16_dinov3', 'vit_large_patch16_dinov3_qkvb', 'vit_small_patch16_dinov3', 'vit_small_patch16_dinov3_qkvb', 'vit_small_plus_patch16_dinov3', 'vit_small_plus_patch16_dinov3_qkvb']


In [ ]:
# =============================================================================
# 1. 演算の超高速化（NVIDIA GPUの力を120%引き出す）
# =============================================================================
# TF32（TensorFloat-32）を許可：
# 精度をほぼ維持しつつ、A100やH100などのGPUで計算速度を数倍に加速させる設定です。
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
# cuDNNベンチマーク：現在の入力サイズに最適な計算アルゴリズムをGPUに自動選択させます。
torch.backends.cudnn.benchmark = True
# 行列演算の精度指定：「high」にすることで速度と精度のバランスを最高レベルに保ちます。
torch.set_float32_matmul_precision("high")

# =============================================================================
# 2. CFG：基本設定とパス（マシンの居場所と名前）
# =============================================================================
class CFG:
    CREATE_SUBMISSION = True    # 予測ファイル（submission.csv）を作るかどうか
    USE_TQDM        = False     # 進捗バー（tqdm）を表示するか（Kaggleログを汚さないためオフ）
    PRETRAINED      = True      # 学習済みモデル（世界中の画像で修行した脳みそ）を使う
    BASE_PATH       = '/kaggle/input/csiro-biomass'
    SEED            = 82947501  # 乱数シード。結果の再現性を確保するための「おまじない」
    N_FOLDS         = 5         # データを5分割して学習（5回戦やって平均をとる）

    # --- モデルの選択 ---
    # ViT Huge: DINOv3の中でも「最大・最強」のモデルを選択。
    MODEL_NAME      = 'vit_huge_plus_patch16_dinov3.lvd1689m'
    DINO_GRAD_CHECKPOINTING = True # 巨大モデルを動かすため、メモリを節約する技術をオン

    IMG_SIZE        = 512       # 入力画像サイズ。デカいほど精細だがメモリを食う

    # =============================================================================
    # 3. 学習のコントロール（ここが現場の「腕」！）
    # =============================================================================
    # バッチサイズ 1 ＆ 勾配蓄積 4：
    # Hugeモデルは1枚でGPUメモリを食い尽くすので「1枚ずつ計算」し、
    # 4回分結果を溜めてから学習（更新）することで、実質「4枚まとめ学習」と同等の安定感を得る。
    BATCH_SIZE      = 1
    GRAD_ACC        = 4 
    
    NUM_WORKERS     = 4         # CPUの腕の数。画像のロードを4並列で行う
    
    # 周回数と準備運動：
    # 1周（Epoch 1）で仕留める短期決戦。ただし、最初の3周（Warmup 3）は
    # 非常に低い学習率で「慣らし運転」をして、脳みそが壊れるのを防ぐ。
    EPOCHS          = 1
    FREEZE_EPOCHS   = 0         # 脳みそを固めておく期間（今回は最初からフル稼働）
    WARMUP_EPOCHS   = 3

    # 学習率（歩幅）：
    # 出口（LR_REST）は 0.001、脳みそ本体（LR_BACKBONE）は 0.0005。
    # すでに賢い本体は「少しだけ変える」のがコツ。
    LR_REST         = 1e-3
    LR_BACKBONE     = 5e-4
    
    WD              = 1e-2      # Weight Decay（荷重減衰）：過学習を防ぐためのペナルティ
    EMA_DECAY       = 0.9       # 指数移動平均：モデルの状態を0.9の割合で「じわっと」更新し、安定させる

    # =============================================================================
    # 4. ターゲットと重み（何を重視して当てるか）
    # =============================================================================
    # 直接当てる項目と、計算で出す項目（Dry_Clover_g, Dry_Dead_g）を整理
    TARGET_COLS     = ['Dry_Total_g', 'GDM_g', 'Dry_Green_g']
    
    # R2_WEIGHTS（決定係数の重み）：
    # [Green, Dead, Clover, GDM, Total] の順。
    # Total（合計）の重みを「0.5」と非常に高く設定している。
    # つまり「合計さえ当たっていれば、他が少しズレても評価してやる」という戦略。
    R2_WEIGHTS      = np.array([0.1, 0.1, 0.1, 0.2, 0.5])
    
    # LOSS_WEIGHTS（学習時のペナルティ）：
    # 逆に学習中は「Green, Dead, Clover」の内訳をしっかり見ろ（0.1ずつ）、
    # 合計は0.0でいい（内訳が合えば合計は自ずと合うから）という指示。
    LOSS_WEIGHTS    = np.array([0.1, 0.1, 0.1, 0.0, 0.0])

    DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Device : cuda
Backbone: vit_huge_plus_patch16_dinov3.lvd1689m | Input: 512
Freeze Epochs: 0 | Warmup: 3
EMA Decay: 0.9 | Grad Acc: 4


In [ ]:
# =============================================================================
# 1. 決定係数 (R2 Score): 統計学でおなじみの「当てはまりの良さ」
# =============================================================================
def weighted_r2_score(y_true, y_pred):
    """
    各項目（緑、枯れ等）のR2を計算し、CFGで決めた重みを掛けて合計する。
    R2 = 1 - (残差平方和 / 全平方和)
    """
    weights = CFG.R2_WEIGHTS
    r2_scores = []
    for i in range(y_true.shape[1]):
        yt = y_true[:, i]; yp = y_pred[:, i]
        ss_res = np.sum((yt - yp) ** 2)              # AIのミス
        ss_tot = np.sum((yt - np.mean(yt)) ** 2)     # データのバラつき
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        r2_scores.append(r2)
    
    # 重み付け平均を出し、コンペのスコアに近い指標を作る
    weighted = np.sum(np.array(r2_scores) * weights) / np.sum(weights)
    return weighted, np.array(r2_scores)

def weighted_r2_score_global(y_true, y_pred):
    """
    【グローバルR2】
    全サンプル・全項目を1つの大きなベクトルとして扱い、モデル全体の「総合力」を測る。
    個別のR2が良くても、全体でバイアス（偏り）がないかをチェックするために重要。
    """
    weights = CFG.R2_WEIGHTS
    flat_true = y_true.reshape(-1)
    flat_pred = y_pred.reshape(-1)
    # 重みを行列サイズに合わせて並べる
    w = np.tile(weights, y_true.shape[0])
    mean_w = np.sum(w * flat_true) / np.sum(w)
    ss_res = np.sum(w * (flat_true - flat_pred) ** 2)
    ss_tot = np.sum(w * (flat_true - mean_w) ** 2)
    global_r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return global_r2

# =============================================================================
# 2. Analyze Errors: 「なぜ間違えたか」を画像レベルで特定する
# =============================================================================
def analyze_errors(val_df, y_true, y_pred, targets, top_n=5):
    """
    誤差（AbsErr）が大きかった最悪の5件をリストアップする。
    画像パスや地域（State）を表示することで、「影が濃い画像に弱い」
    「特定の州のデータが苦手」といったAIの弱点を見抜き、次の改善に繋げる。
    """
    print(f'\n--- Top {top_n} High Loss Samples per Target ---')
    y_true = np.array(y_true); y_pred = np.array(y_pred)
    
    for i, target in enumerate(targets):
        errors = np.abs(y_true[:, i] - y_pred[:, i])
        top_indices = np.argsort(errors)[::-1][:top_n] # 誤差が大きい順
        
        for idx in top_indices:
            # 画像名や州を表示して、データの中身を人間が確認しにいくための情報
            path = os.path.basename(val_df.iloc[idx]['image_path'])
            state = val_df.iloc[idx].get('State', 'NA')
            print(f'Target: {target} | Image: {path} | State: {state} | Err: {errors[idx]:.4f}')

# =============================================================================
# 3. Compare Train/Val: 学習データとテストデータに「ズレ」がないか確認
# =============================================================================
def compare_train_val(tr_df, val_df, targets):
    """
    【健康診断】
    学習用(train)と検証用(val)で、草の重さの平均や分布が違わないかチェック。
    もしズレていたら、学習しても検証でスコアが出ない「共変量シフト」が起きている証拠。
    """
    for t in targets:
        tr = tr_df[t].dropna(); val = val_df[t].dropna()
        # ここでKDEプロット（分布図）を描き、視覚的に「分布の重なり」を確認する
        # ... (SNS.KDEPLOTなどで可視化) ...

# =============================================================================
# 4. Biomass Loss: 学習中の「反省の深さ」を決める数式
# =============================================================================
def biomass_loss(outputs, labels, w=None):
    """
    AIの学習（反省）に使う関数。
     Huber Loss (SmoothL1Loss) を採用しているのがポイント。
    """
    huber = nn.SmoothL1Loss(beta=5.0) # 誤差が小さい時は二乗、大きい時は直線的に評価
    
    # 各項目（Green, Dead, Clover, GDM, Total）のロスを計算
    l_green  = huber(outputs[2], labels[:,0])
    l_dead   = huber(outputs[4], labels[:,1])
    l_clover = huber(outputs[3], labels[:,2])
    l_gdm    = huber(outputs[1], labels[:,3])
    l_total  = huber(outputs[0], labels[:,4])

    losses = torch.stack([l_green, l_dead, l_clover, l_gdm, l_total])
    
    # CFGで設定したLOSS_WEIGHTS（内訳重視）を掛けて、最終的な「反省量」を決定
    w = torch.as_tensor(w, device=losses.device)
    return (losses * (w / w.sum())).sum()

In [ ]:
# =============================================================================
# 1. Transforms: 画像の「水増し（Augmentation）」と「規格化」
# =============================================================================
def get_train_transforms():
    """
    【学習用】AIに「いろんな角度・状態の草」を見せて、打たれ強くする。
    """
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.HorizontalFlip(p=0.5), # 50%の確率で左右反転
        A.VerticalFlip(p=0.5),   # 上下反転
        A.RandomRotate90(p=0.5), # 90度回転
        # わずかな回転（-10度〜10度）を加えて、撮影時のカメラの傾きを再現
        A.Rotate(limit=(-10, 10), p=0.3, interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT_101),
        # 明るさ、コントラスト、色合いをわずかに変える（天候の違いをシミュレート）
        A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05, p=0.5),
        # 【重要】Normalize：画像を世界標準（ImageNet）の平均・標準偏差に合わせる
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2() # PyTorchが計算できる形（テンソル）に変換
    ], p=1.0)

def get_tta_transforms(mode=0):
    """
    【推論用（TTA）】予測時に、あえて画像を反転させて、その平均をとる。
    「たまたま角度が悪くて外れた」というミスを防ぐための「複数回テスト」用。
    """
    # ...（modeによって「そのまま」「反転」「90度回転」を切り替える）...
    # ※ Albumentationsを使って「きっちり90度」回すための精密な設定になっています。
    # mode 0: original
    # mode 1: hflip
    # mode 2: vflip
    # mode 3: rotate90
    transforms_list = [
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
    ]
    
    if mode == 1:
        transforms_list.append(A.HorizontalFlip(p=1.0))
    elif mode == 2:
        transforms_list.append(A.VerticalFlip(p=1.0))
    elif mode == 3:
        transforms_list.append(A.RandomRotate90(p=1.0)) # RandomRotate90 with p=1.0 rotates 90, 180, 270 randomly? 
        # Albumentations RandomRotate90 rotates by 90, 180, 270. 
        # Reference uses transforms.RandomRotation([90, 90]) which is exactly 90 degrees.
        # To match exactly 90 degrees in Albumentations, we might need Rotate(limit=(90,90), p=1.0)
        # But RandomRotate90 is standard TTA. Let's use Rotate(limit=(90,90)) to be precise if that's what reference does.
        # Reference: transforms.RandomRotation([90, 90]) -> rotates by exactly 90 degrees.
        transforms_list.append(A.Rotate(limit=(90, 90), p=1.0, interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT_101))

    transforms_list.extend([
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
    
    return A.Compose(transforms_list, p=1.0)
# =============================================================================
# 2. clean_image: 「ゴミ掃除」の職人芸
# =============================================================================
def clean_image(img):
    """
    このコードが凄いのはここです！データ特有の「汚れ」を手動で消しています。
    """
    # ① 下部10%をカット：カメラの枠やゴミが写り込みやすい一番下を切り捨てる
    h, w = img.shape[:2]
    img = img[0:int(h*0.90), :] 

    # ② オレンジ色の「日付スタンプ」を消す：
    # AIは賢すぎるので、日付の数字で重さを推測しようとしてしまいます（カンニング防止）。
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV) # 色の判別がしやすいHSV空間に変換
    lower = np.array([5, 150, 150])  # オレンジ色の下限
    upper = np.array([25, 255, 255]) # オレンジ色の上限
    mask = cv2.inRange(hsv, lower, upper) # オレンジ色の部分だけ「白」にしたマスク作成

    # マスクを太らせて、文字の縁までカバーする
    mask = cv2.dilate(mask, np.ones((3,3), np.uint8), iterations=2)

    # Inpaint（インペイント）：オレンジの文字を、周りの草の色で塗りつぶして「消滅」させる
    if np.sum(mask) > 0:
        img = cv2.inpaint(img, mask, 3, cv2.INPAINT_TELEA)

    return img

# =============================================================================
# 3. BiomassDataset: 左右のカメラ画像を別々に処理する
# =============================================================================
class BiomassDataset(Dataset):
    """
    このDatasetは、1つの画像を「真ん中でパカッと割り」ます。
    ステレオカメラ（左右2つのレンズ）で撮られた画像を、別々の入力として扱います。
    """
    def __getitem__(self, idx):
        # ...（読み込みとclean_image実行）...
        mid = w // 2
        left = img[:, :mid]   # 左半分の画像
        right = img[:, mid:]  # 右半分の画像
        
        # 左右それぞれに水増し（回転や色補正）をかけてAIに渡す
        left = self.transform(image=left)['image']
        right = self.transform(image=right)['image']
        
        return left, right, label

In [ ]:
# =============================================================================
# LocalMambaBlock: 次世代の高速情報処理ブロック
# =============================================================================
class LocalMambaBlock(nn.Module):
    """
    「Mamba」という最新理論（線形計算量モデル）のエッセンスを取り入れたブロック。
    Transformerの弱点である「計算の重さ」を克服しつつ、重要な情報を抽出します。
    """
    def __init__(self, dim, kernel_size=5, dropout=0.0):
        super().__init__()
        # 1. LayerNorm: データの数値を「平均0、分散1」付近に整えて、学習を安定させる
        self.norm = nn.LayerNorm(dim)
        
        # 2. Depthwise Conv (1D): 
        # 「隣り合うトークン（画像の断片）」同士を混ぜ合わせる。
        # groups=dim にすることで、計算量を抑えつつ各次元の特徴を独立して抽出。
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=kernel_size, 
                               padding=kernel_size // 2, groups=dim)
        
        # 3. Gating Mechanism (門番): 
        # ここがMambaのキモ。情報の「重要度」を判定するフィルターを作る。
        self.gate = nn.Linear(dim, dim)
        
        # 4. Projection: 最終的に情報を整えて出力する
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        # x: (Batch, Tokens, Dim) 
        # ここでのトークンは、DINOが細かく分解した画像のパッチ（断片）のこと
        shortcut = x # 1. 元の情報を取っておく（あとで足すことで情報の消失を防ぐ）
        
        x = self.norm(x)
        
        # 2. ゲート（門）を開ける: 
        # sigmoid関数を使って、0.0〜1.0の間で「どの情報を残すか」を決定する
        g = torch.sigmoid(self.gate(x))
        x = x * g # 重要な情報だけを通過させる（情報の選別）
        
        # 3. 空間情報の混合:
        # 画像を横一列に並んだデータとして扱い、1D Convで前後のつながりを確認する
        x = x.transpose(1, 2)  # (B, N, D) -> (B, D, N) : 計算のために並べ替え
        x = self.dwconv(x)
        x = x.transpose(1, 2)  # (B, D, N) -> (B, N, D) : 元に戻す
        
        # 4. 仕上げ
        x = self.proj(x)
        x = self.drop(x)
        
        # 5. 残差接続: 
        # 「新しく学んだこと」に「元の情報」を足して出力する
        return shortcut + x

In [ ]:
# =============================================================================
# BiomassModel: DINOv3 + Mamba Fusion + 3-Stream Heads
# =============================================================================
class BiomassModel(nn.Module):
    def __init__(self, model_name, pretrained=True, backbone_path=None):
        super().__init__()
        
        # 1. バックボーン（脳みそ）の準備
        # global_pool='' にするのがプロの技。
        # 画像を「1枚の要約データ」にせず、「バラバラの断片（パッチ）」のまま取り出します。
        # これにより、後段のMambaが「画像のどこに何があるか」を細かく分析できます。
        self.backbone = timm.create_model(self.model_name, pretrained=False, num_classes=0, global_pool='')
        
        # 2. メモリ節約術 (Gradient Checkpointing)
        # ViT-Hugeは巨大すぎて、これがないと普通のGPUでは爆発（Out of Memory）します。
        # 計算を少し遅くする代わりに、メモリ消費を半分に抑える魔法の設定です。
        if hasattr(self.backbone, 'set_grad_checkpointing') and CFG.DINO_GRAD_CHECKPOINTING:
            self.backbone.set_grad_checkpointing(True)
            
        nf = self.backbone.num_features # 特徴量の次元数（Hugeなら1280次元など）
        
        # 3. Mamba Fusion Neck (合体ロボの結合部)
        # 左右のカメラから来た別々の情報を、ここで1つに混ぜ合わせます。
        # さっきのLocalMambaBlockを2段重ねにして、情報の密度を高めます。
        self.fusion = nn.Sequential(
            LocalMambaBlock(nf, kernel_size=5, dropout=0.1),
            LocalMambaBlock(nf, kernel_size=5, dropout=0.1)
        )
        
        # 4. 出口（Head）の設計：Softplusがポイント
        # 「緑」「クローバー」「枯れ草」それぞれに専用の出口を作ります。
        # nn.Softplus() を使うことで、予測値が必ず「正（0以上）」になるようにします。
        # 「草の重さがマイナス」という物理的な矛盾を数式レベルで封じ込めています。
        self.head_green_raw  = nn.Sequential(
            nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), nn.Softplus()
        )
        # ... (clover, dead も同様) ...

    def forward(self, left, right):
        """
        [推論の流れ]
        1. 左右の画像を別々にDINOv3に入力
        2. 得られた「画像の断片(Tokens)」を横に繋げる (B, 2N, D)
        3. Mambaで左右の情報の「差」や「共通点」を抽出
        4. 全体を平均(Pooling)して、1つのベクトルにする
        5. 各出口から予測を出し、足し算で合計を出す
        """
        x_l = self.backbone(left)
        x_r = self.backbone(right)
        
        # 左右合体！
        x_cat = torch.cat([x_l, x_r], dim=1)
        
        # Mambaで情報の化学反応を起こす
        x_fused = self.fusion(x_cat)
        
        # プーリング：(Batch, Tokens, Dim) -> (Batch, Dim)
        x_pool = self.pool(x_fused.transpose(1, 2)).flatten(1)
        
        # 最終予測：内訳を出してから足す
        green  = self.head_green_raw(x_pool)
        clover = self.head_clover_raw(x_pool)
        dead   = self.head_dead_raw(x_pool)
        
        # 【重要】生物学的足し算：内訳を足して合計にする
        gdm    = green + clover
        total  = gdm + dead
        
        return total, gdm, green, clover, dead

In [ ]:
# =============================================================================
# 1. 脳みその「固定 / 解放」スイッチ
# =============================================================================
def set_backbone_requires_grad(model: BiomassModel, requires_grad: bool):
    """
    バックボーン（DINOv3）の重みを更新するかどうかを切り替える。
    - Falseにすると「脳みそを固める（Freeze）」：学習させず、今の知識を維持。
    - Trueにすると「脳みそを鍛える（Unfreeze）」：新しいデータで知識を上書き。
    初期段階で手足（Heads）だけを先に鍛えたい時によく使われます。
    """
    for p in model.backbone.parameters():
        p.requires_grad = requires_grad


# =============================================================================
# 2. オプティマイザ（最適化器）の構築：二段階学習率
# =============================================================================
def build_optimizer(model: BiomassModel):
    """
    「すでに天才（脳）」と「これから学ぶ新人（手足）」で、教えるスピードを変える。
    """
    # 1. バックボーンのパラメータIDを特定して、仲間分けの準備をする
    backbone_ids = {id(p) for p in model.backbone.parameters()}
    
    # 2. 全パラメータを「脳みそ側」と「それ以外（Heads, Fusion）」に分離
    backbone_params = []
    rest_params = []
    
    for p in model.parameters():
        if p.requires_grad: # 学習対象のパラメータのみピックアップ
            if id(p) in backbone_ids:
                backbone_params.append(p) # 脳みそグループ
            else:
                rest_params.append(p)     # 手足グループ（Fusion層や予測出口）
    
    # AdamW（非常に優秀な最適化アルゴリズム）に、別々の歩幅（学習率）を設定
    return optim.AdamW([
        # 脳みそ：すでに完成されているので、極小の歩幅（LR_BACKBONE）で慎重に微調整
        {'params': backbone_params, 'lr': CFG.LR_BACKBONE, 'weight_decay': CFG.WD},
        # 手足：一から学ぶ必要があるため、大きな歩幅（LR_REST）で大胆に学習
        {'params': rest_params,     'lr': CFG.LR_REST,     'weight_decay': CFG.WD},
    ])



# =============================================================================
# 3. スケジューラ：学習のリズム（歩幅の変化）を管理
# =============================================================================
def build_scheduler(optimizer):
    """
    学習が進むにつれて「歩幅（学習率）」をどう変化させるかを定義する。
    """
    def lr_lambda(epoch):
        e = max(0, epoch - 1)
        
        # --- Warmup（準備運動）フェーズ ---
        # 最初の数エポックは、0から徐々に歩幅を広げていく。
        # 巨大モデルがいきなり全力で走り出してパニック（勾配爆発）を起こすのを防ぐ。
        if e < CFG.WARMUP_EPOCHS:
            return float(e + 1) / float(max(1, CFG.WARMUP_EPOCHS))
        
        # --- Cosine Annealing（コサイン減速）フェーズ ---
        # 準備運動が終わったら、コサイン曲線のグラフに沿って、最後に向けて徐々に歩幅を小さくする。
        # ゴール（正解）に近づくほど慎重に歩を進め、ピンポイントで最適解を射抜くための戦略。
        progress = (e - CFG.WARMUP_EPOCHS) / float(max(1, CFG.EPOCHS - CFG.WARMUP_EPOCHS))
        return 0.5 * (1.0 + math.cos(math.pi * progress))
        
    return LambdaLR(optimizer, lr_lambda)

In [ ]:
# =============================================================================
# 1. 演算精度と高速化の設定 (Mixed Precision)
# =============================================================================
USE_BF16 = True
# BF16（bfloat16）は、計算速度を上げつつ数値の安定性を保つGoogle発の形式。
# これにより、巨大なモデルの学習がスピードアップし、メモリ消費も抑えられます。
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

# GradScaler: 計算中に数値が小さくなりすぎて「0」になる（勾配消失）のを防ぐスケーラー。
scaler = torch.amp.GradScaler(
    'cuda',
    enabled=(torch.cuda.is_available() and AMP_DTYPE == torch.float16)
)

def autocast_ctx():
    """「ここは高速モードで計算してOK」という範囲を指定するコンテキスト管理"""
    if not torch.cuda.is_available():
        return nullcontext()
    return torch.amp.autocast(device_type='cuda', dtype=AMP_DTYPE)

# =============================================================================
# 2. データの整形ユーティリティ
# =============================================================================
# モデルの出力やラベルの「次元（形）」を、評価用メトリクスが求める形に
# 強制的に整えるガードレール的な関数群です。
def _as_col(x, bs): ... # (B,) を (B,1) に整える
def _ensure_2d_lab(lab, bs): ... # ラベルを必ず2次元にする

def _pred_pack(p_total, p_gdm, p_green, p_clover, p_dead, bs: int) -> torch.Tensor:
    """
    【重要】バラバラに出力された5つの予測値を、
    CFG.ALL_TARGET_COLS の順番通りに横に並べて (B, 5) の行列にパッキングする。
    """
    pg = _as_col(p_green,  bs)
    pd = _as_col(p_dead,   bs)
    pc = _as_col(p_clover, bs)
    pgdm = _as_col(p_gdm,  bs)
    pt = _as_col(p_total,  bs)
    return torch.cat([pg, pd, pc, pgdm, pt], dim=1)

# =============================================================================
# 3. Validation Loop: 実力テスト
# =============================================================================
@torch.inference_mode() # 勾配計算をオフにしてメモリを節約
def valid_epoch(eval_model, loader, device):
    """通常の検証。1枚ずつ画像を見て、どれだけ当たっているか集計する。"""
    eval_model.eval() # 評価モード（Dropoutなどをオフにする）
    # ... (集計用のメモリ確保) ...
    for l, r, lab in loader:
        with autocast_ctx(): # 高速精度モードで予測
            p_total, p_gdm, p_green, p_clover, p_dead = eval_model(l, r)
            loss = biomass_loss(...)
        # 予測結果をCPUに送って保存
        batch_pred = _pred_pack(...).float().cpu()
        preds_cpu[offset:offset+bs]  = batch_pred
    # 全データ終了後、R2スコアを計算して返す
    return loss, global_r2, avg_r2, per_r2, ...

# =============================================================================
# 4. TTA Validation: 「慎重な」実力テスト
# =============================================================================
@torch.inference_mode()
def valid_epoch_tta(eval_model, loaders, device):
    """
    Test Time Augmentation (TTA) 版。
    「そのままの画像」「左右反転した画像」など、複数のパターンの予測値を出し、
    その平均をとることで、予測のブレを最小限に抑える（スコアが伸びる技）。
    """
    # 複数のローダー（画像加工パターン別）の予測を足し合わせる
    for tta_i, loader in enumerate(loaders):
        for l, r, lab in loader:
            batch_pred = _pred_pack(...)
            preds_sum[offset:offset+bs] += batch_pred # 予測を累積
    avg_preds = (preds_sum / len(loaders)).numpy() # 最後に平均をとる
    return ...

# =============================================================================
# 5. Training Loop: 本番の学習
# =============================================================================
def train_epoch(model, loader, opt, scheduler, device, ema=None):
    """AIに「反省」を促し、重みを更新させるメインループ。"""
    model.train() # 学習モード
    
    for i, (l, r, lab) in enumerate(loader):
        # 1. 前方伝播: 予測を出す
        with autocast_ctx():
            outputs = model(l, r)
            loss = biomass_loss(outputs, lab, w=CFG.LOSS_WEIGHTS)
            loss = loss / CFG.GRAD_ACC # 勾配蓄積のための調整

        # 2. 逆伝播: どのパラメータがどれくらい間違えたか計算
        if scaler.is_enabled():
            scaler.scale(loss).backward()
        else:
            loss.backward()

        # 3. 重み更新: 指定回数(GRAD_ACC)溜まったら実際に一歩進む
        do_step = ((i + 1) % CFG.GRAD_ACC == 0) or ((i + 1) == len(loader))
        if do_step:
            if scaler.is_enabled():
                scaler.unscale_(opt) # 精度調整を戻す
                # 勾配クリッピング：歩幅が大きすぎたら強制的に抑える（爆発防止）
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            # EMA更新: 指数移動平均モデルを更新（安定性を高める）
            if ema is not None:
                ema.update(model)

            opt.zero_grad(set_to_none=True) # 溜まったミスをリセット

    scheduler.step() # 学習率を計画通りに変更
    return running_loss / len(loader.dataset)

In [ ]:
# =============================================================================
# 1. データの整理と「リーク」対策
# =============================================================================
# df_wide = df_long.pivot(...) 
# ターゲット（緑、枯れなど）を横一列に並べ替え、1枚の画像に対して1行のデータに整えます。

# StratifiedGroupKFold: ここがプロの統計処理です！
# 「同じ日(Sampling_Date)に撮った写真は、学習と検証に分けない（Group）」
# 「各州(State)の比率は均等に分ける（Stratified）」
# これにより、AIが「日付を覚える」というカンニングを防ぎ、真の実力を測れるようにします。



# =============================================================================
# 2. Foldごとの学習（交差検証）
# =============================================================================
# for fold, (tr_idx, val_idx) in enumerate(sgkf.split(...)):
# データを例えば5つに分け、そのうち4つで学び、1つでテストする…を繰り返します。
# これにより、データの偏りに強い「頑健なモデル」が作れます。

# =============================================================================
# 3. 段階的な教育（Freeze / Unfreeze 戦略）
# =============================================================================
# set_backbone_requires_grad(model, False)
# ↓（数エポック後）
# set_backbone_requires_grad(model, True)
# 最初は「脳みそ（DINO）」を固定して、新しく作った「手足（Heads）」だけを馴染ませます。
# その後、全体を解禁して「微調整（Fine-tuning）」に入るという二段構えです。

# =============================================================================
# 4. 指数移動平均 (EMA) による安定化
# =============================================================================
# ema = ModelEmaV2(model, decay=CFG.EMA_DECAY)
# 毎秒の学習結果だけでなく、「これまでの平均的な知能」を別途保存しておきます。
# 評価（valid_epoch_tta）にはこのEMAモデルを使うことで、スコアが安定します。

# =============================================================================
# 5. Early Stopping（早期終了）と保存
# =============================================================================
# if global_r2 > best_global_r2:
#     torch.save(...) # 過去最高スコアが出たら保存
# else:
#     patience += 1 # スコアが伸びなくなったら「忍耐」を消費

# 決められた回数(PATIENCE)スコアが更新されなければ、
# 「これ以上は時間の無駄（過学習の恐れ）」と判断して学習を切り上げます。

In [ ]:
# =============================================================================
# 4. TTA Transforms: 「複数視点」による慎重な予測
# =============================================================================
def get_tta_transforms(num_transforms):
    """
    Test Time Augmentation (TTA) の設定。
    1つの画像を「そのまま」「左右反転」「上下反転」「両方反転」の4パターンで予測し、
    その平均をとることで、1枚の画像に対する予測の「ブレ」を抑えます。
    """
    normalize = A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    to_tensor = ToTensorV2()
    
    all_tta_transforms = [
        A.Compose([A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE), normalize, to_tensor]), # そのまま
        A.Compose([A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE), A.HorizontalFlip(p=1.0), normalize, to_tensor]), # 左右反転
        A.Compose([A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE), A.VerticalFlip(p=1.0), normalize, to_tensor]),   # 上下反転
        A.Compose([A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE), A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0), normalize, to_tensor]), # 両方
    ]
    return all_tta_transforms[:num_transforms]

# =============================================================================
# 5. Create Test Dataset: テスト用データの読み込み
# =============================================================================
class BiomassTestDataset(Dataset):
    """
    テスト画像を読み込み、学習時と同じく「左右に分割」してAIに渡す準備をする。
    """
    def __init__(self, img_dir):
        self.img_dir = img_dir
        self.paths = sorted([os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.filenames = [os.path.basename(p) for p in self.paths]
    
    def __len__(self): return len(self.paths)
    
    def __getitem__(self, idx):
        path = self.paths[idx]
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = clean_image(img) # 学習時と同じ「ゴミ掃除」を適用
        mid = img.shape[1] // 2
        left, right = img[:, :mid].copy(), img[:, mid:].copy()
        return left, right, self.filenames[idx]

# =============================================================================
# 8. Run Inference with TTA: 予測の実行とアンサンブル
# =============================================================================
@torch.no_grad() # 予測モード。メモリを節約
def predict_with_tta(model, left_np, right_np, tta_transforms):
    """1つのモデルを使って、TTA（4つの視点）の平均予測値を出す。"""
    all_tta_preds = []
    for tfm in tta_transforms:
        left_t = tfm(image=left_np)['image'].unsqueeze(0).to(CFG.DEVICE)
        right_t = tfm(image=right_np)['image'].unsqueeze(0).to(CFG.DEVICE)
        total, gdm, green, clover, dead = model(left_t, right_t)
        all_tta_preds.append([total.item(), gdm.item(), green.item()])
    return np.mean(all_tta_preds, axis=0) # TTAの平均

def run_inference():
    """
    全Foldのモデルを順番に読み込み、それらすべての予測を足し合わせる（アンサンブル）。
    """
    dataset = BiomassTestDataset(CFG.TEST_IMAGE_DIR)
    loader = DataLoader(dataset, batch_size=1, shuffle=False)
    tta_transforms = get_tta_transforms(CFG.TTA_STEPS)
    accumulated_preds = np.zeros((len(dataset), 3), dtype=np.float32)

    folds_to_use = getattr(CFG, 'FOLDS_TO_TRAIN', list(range(CFG.N_FOLDS)))
    successful_folds = 0
    
    for fold in folds_to_use:
        # 各Foldの「一番良かった時の重み」をロード
        model = BiomassModel(CFG.MODEL_NAME, pretrained=False)
        weight_path = os.path.join(CFG.MODEL_DIR, f'best_model_fold{fold}.pth')
        if not os.path.exists(weight_path): continue
        
        model.load_state_dict(torch.load(weight_path, map_location='cpu'))
        model.to(CFG.DEVICE).eval()
        
        for i, (left, right, _) in enumerate(loader):
            pred = predict_with_tta(model, left[0].numpy(), right[0].numpy(), tta_transforms)
            accumulated_preds[i] += pred # 各Foldの予測を累積
        successful_folds += 1

    return accumulated_preds / successful_folds, dataset.filenames # 平均をとって最終予測

# =============================================================================
# 9. Post-Process: 数学的・生物学的整合性のチェック
# =============================================================================
def postprocess_predictions(preds_direct):
    """
    AIが出した [total, gdm, green] から、引き算で [dead, clover] を算出する。
    np.maximum(0, ...) を使うことで、「重さがマイナスになる」という計算上のエラーを
    物理的にありえない数値として修正（0にクリップ）します。
    """
    pred_total, pred_gdm, pred_green = preds_direct[:, 0], preds_direct[:, 1], preds_direct[:, 2]
    pred_clover = np.maximum(0, pred_gdm - pred_green)
    pred_dead = np.maximum(0, pred_total - pred_gdm)
    
    return np.stack([pred_green, pred_dead, pred_clover, pred_gdm, pred_total], axis=1)

# =============================================================================
# 10. Create Submission: 提出用CSVの作成
# =============================================================================
def create_submission(predictions, filenames):
    """
    AIの予測結果を、Kaggleが指定する「long format（縦持ち形式）」に変換し、
    sample_idと紐づけて保存する。
    """
    test_df = pd.read_csv(CFG.TEST_CSV)
    # パス形式の調整（"test/" プレフィックスの付与など）
    # ... (パスの一致確認と修正ロジック) ...
    
    # データを縦長に変形 (Melt)
    preds_long = pd.DataFrame(predictions, columns=CFG.ALL_TARGET_COLS).assign(image_path=filenames).melt(...)
    
    # 元のテストリストと結合して、最終的な提出ファイルを作成
    submission = pd.merge(test_df[['sample_id', 'image_path', 'target_name']], preds_long, on=['image_path', 'target_name'], how='left')
    submission.to_csv('submission70.csv', index=False)
    return submission

✓ TTA transforms defined (1 views)
✓ Test dataset class defined

STARTING INFERENCE
Folds requested for inference: [0, 1, 2, 3, 4]

Processing Fold 0...


Fold 0: 100%|██████████| 1/1 [00:01<00:00,  1.60s/it]



Processing Fold 1...


Fold 1: 100%|██████████| 1/1 [00:01<00:00,  1.45s/it]



Processing Fold 2...


...


Processing Fold 3...


Fold 3: 100%|██████████| 1/1 [00:01<00:00,  1.61s/it]



Processing Fold 4...


Fold 4: 100%|██████████| 1/1 [00:01<00:00,  1.60s/it]



Inference complete. Successfully used 5 fold(s) out of 5 requested.

Post-processing predictions...
✓ Post-processing complete
  Output shape: (1, 5)

Prediction statistics:
  Dry_Green_g    : mean=26.58, std=0.00, min=26.58, max=26.58
  Dry_Dead_g     : mean=32.42, std=0.00, min=32.42, max=32.42
  Dry_Clover_g   : mean=0.26, std=0.00, min=0.26, max=0.26
  GDM_g          : mean=26.84, std=0.00, min=26.84, max=26.84
  Dry_Total_g    : mean=59.26, std=0.00, min=59.26, max=59.26

CREATING SUBMISSION FILE

Test CSV loaded: 5 rows
Sample image_path from test.csv: test/ID1001187975.jpg
Sample filename from predictions: ID1001187975.jpg
Corrected path format: test/ID1001187975.jpg

Wide format predictions:
              image_path  Dry_Green_g  Dry_Dead_g  Dry_Clover_g      GDM_g  \
0  test/ID1001187975.jpg    26.581457    32.41909      0.262262  26.843719   

   Dry_Total_g  
0     59.26281  

Long format predictions (first 10 rows):
              image_path   target_name     target
0  test

In [ ]:
# =============================================================================
# 1. Configuration: 2つの知能の比率
# =============================================================================
# DINO（さっきまで見ていたもの）と、もう一つの強力なAI「SigLIP」の予測結果を混ぜます。
# ここでは DINO(52%) : SigLIP(48%) の割合でブレンドしています。
W_SIGLIP = 0.48
W_DINO   = 0.52

# =============================================================================
# 2. enforce_mass_balance: 「生物学的・物理的な整合性」の強制
# =============================================================================
def enforce_mass_balance(df_wide, fixed_clover=None):
    """
    【最重要】直交射影（Orthogonal Projection）という数学を用いて、
    AIの予測値が以下の物理法則を100%満たすように強制修正します。
    1. Dry_Green_g + Dry_Clover_g = GDM_g (緑とクローバーを足すとGDMになる)
    2. GDM_g + Dry_Dead_g = Dry_Total_g (GDMと枯れ草を足すと合計になる)
    """
    ordered_cols = ['Dry_Green_g', 'Dry_Clover_g', 'Dry_Dead_g', 'GDM_g', 'Dry_Total_g']
    Y = df_wide[ordered_cols].values.T
    
    if fixed_clover:
        # クローバー(Clover)はDINOが一番正確なので、そこを固定して他を微調整するモード
        clover_fixed = Y[1, :].copy()
        Y[3, :] = Y[0, :] + clover_fixed # GDMを再計算
        Y[4, :] = Y[3, :] + Y[2, :]      # Totalを再計算
        Y_reconciled = Y
    else:
        # 「最小二乗法」に似た計算で、予測値からの変化を最小にしつつ方程式を満たす点を計算
        C = np.array([
            [1, 1, 0, -1,  0], # 式1の係数
            [0, 0, 1,  1, -1]  # 式2の係数
        ])
        C_T = C.T
        inv_CCt = np.linalg.inv(C @ C_T)
        P = np.eye(5) - C_T @ inv_CCt @ C # 射影行列Pの作成
        Y_reconciled = P @ Y
    
    Y_reconciled = np.maximum(0, Y_reconciled.T) # 負の値を0にする
    df_out = df_wide.copy()
    df_out[ordered_cols] = Y_reconciled
    return df_out

# =============================================================================
# 3. robust_ensemble: 知能のブレンド
# =============================================================================
def robust_ensemble(file_paths, weights):
    """
    2つのモデル（SigLIPとDINO）を読み込み、それぞれの「得意」を活かして合体させる。
    """
    # 1. データの読み込みとIDの整列
    # ... (CSVをロードし、サンプルIDが一致しているか確認) ...

    # 2. ターゲットごとの戦略
    # Clover（クローバー）はDINOの精度が非常に高いため、混ぜずにDINOの値を100%採用。
    # それ以外（Green, Dead, Totalなど）は、設定した重み（52:48）で平均をとる。
    
    # 3. Mass Balanceの適用
    # 合体させた予測値は、足し算の整合性が崩れている可能性があるため、
    # さっきの enforce_mass_balance に通して「正しい物理法則」を叩き込む。
    
    # 4. 最終フォーマットへの変換
    # ... (Wide形式からKaggle指定のLong形式に戻す) ...
    return final_submission

--- Starting Ensemble ---
Weights: {'siglip': 0.48, 'dino': 0.52}
NOTE: Using DINO-only for Dry_Clover_g (better detection)
Loaded siglip: 5 rows
Loaded dino: 5 rows
Using DINO-only for Dry_Clover_g
Weighted average complete (DINO-only for Clover).
Applying Mass Balance Constraints (Dry_Clover_g fixed to DINO values)...

Success! Saved to submission.csv
                    sample_id     target
0  ID1001187975__Dry_Clover_g   0.000000
1    ID1001187975__Dry_Dead_g  31.982701
2   ID1001187975__Dry_Green_g  28.040880
3   ID1001187975__Dry_Total_g  60.023581
4         ID1001187975__GDM_g  28.040880

Stats:
count     5.000000
mean     29.617609
std      21.285510
min       0.000000
25%      28.040880
50%      28.040880
75%      31.982701
max      60.023581
Name: target, dtype: float64
